In [12]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20")

Q3_FILES = [
    {
        "path": ROOT / "Yifeng Mao" / "data" / "active_lives_1516_london_125.csv",
        "year": 1,
    },
    {
        "path": ROOT / "Yifeng Mao" / "data" / "active_lives_1617_london_125.csv",
        "year": 2,
    },
    {
        "path": ROOT / "Siyan Xin" / "2017~2018" / "2017_data_125_activities.csv",
        "year": 3,
    },
    {
        "path": ROOT / "Siyan Xin" / "2018~2019" / "2018_data_125_activities.csv",
        "year": 4,
    },
    {
        "path": ROOT / "Shuhan Zhao" / "docs" / "1920_london32_stable125.csv",
        "year": 5,
    },
    {
        "path": ROOT / "Shuhan Zhao" / "docs" / "2021_london32_stable125.csv",
        "year": 6,
    },
    {
        "path": ROOT / "Jingyi Hua" / "data" / "processed" / "year7_125activities.csv",
        "year": 7,
    },
    {
        "path": ROOT / "Jingyi Hua" / "data" / "processed" / "year8_125activities.csv",
        "year": 8,
    },
]

OUTPUT_DIR = ROOT / "Shuhan Zhao" / "q3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEIGHT_COL = "wt_final"
MISSING_CODES = [-99, -98, -97, -96, -95, -94, -93, -92, -91]

print("Input file check:")
missing = []
for file_info in Q3_FILES:
    path = Path(file_info["path"])
    ok = path.exists()
    print(f"Year {file_info['year']}: {ok} | {path}")
    if not ok:
        missing.append(str(path))

print("\nOutput directory:")
print(OUTPUT_DIR)

if missing:
    raise FileNotFoundError(
        "Required input files not found:\n" + "\n".join(missing)
    )


Input file check:
Year 1: True | C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Yifeng Mao\data\active_lives_1516_london_125.csv
Year 2: True | C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Yifeng Mao\data\active_lives_1617_london_125.csv
Year 3: True | C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Siyan Xin\2017~2018\2017_data_125_activities.csv
Year 4: True | C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Siyan Xin\2018~2019\2018_data_125_activities.csv
Year 5: True | C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\docs\1920_london32_stable125.csv
Year 6: True | C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\docs\2021_london32_stable125.csv
Year 7: True | C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year7_125activities.csv
Year 8: True | C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year8_125activities.csv

Output directory:
C:\Users\Lenovo\D

In [13]:
AGE9_LABELS = {
    2: "16-24",
    3: "25-34",
    4: "35-44",
    5: "45-54",
    6: "55-64",
    7: "65-74",
    8: "75-84",
    9: "85+",
}


ACTIVITY_LEVEL_LABELS = {
    0: "inactive",
    1: "fairly_active",
    2: "active",
}


MONTHS12_PREFIX = "MONTHS_12_"
DAYS10P60GR_PREFIX = "DAYS10P60GR_"
MEMS7GR_PREFIX = "MEMS7GR_"

In [14]:
def standardise_base_columns(df):
    df = df.copy()

    rename_dict = {}

    if "age16plus" in df.columns and "Age16plus" not in df.columns:
        rename_dict["age16plus"] = "Age16plus"

    if "Month" in df.columns and "month" not in df.columns:
        rename_dict["Month"] = "month"

    df = df.rename(columns=rename_dict)

    return df

In [15]:
def weighted_mean(values, weights):
    mask = values.notna() & weights.notna()

    if mask.sum() == 0:
        return np.nan

    if weights.loc[mask].sum() == 0:
        return np.nan

    return np.average(
        values.loc[mask],
        weights=weights.loc[mask]
    )


def weighted_rate_equal(data, value_col, positive_value, weight_col):
    values = data[value_col]
    weights = data[weight_col]

    mask = values.notna() & weights.notna()

    if mask.sum() == 0:
        return np.nan

    if weights.loc[mask].sum() == 0:
        return np.nan

    indicator = (values.loc[mask] == positive_value).astype(float)

    return np.average(
        indicator,
        weights=weights.loc[mask]
    )


def weighted_rate_positive(data, value_col, weight_col):
    values = data[value_col]
    weights = data[weight_col]

    mask = values.notna() & weights.notna()

    if mask.sum() == 0:
        return np.nan

    if weights.loc[mask].sum() == 0:
        return np.nan

    indicator = (values.loc[mask] > 0).astype(float)

    return np.average(
        indicator,
        weights=weights.loc[mask]
    )


def weighted_category_rate(data, value_col, category_value, weight_col):
    values = data[value_col]
    weights = data[weight_col]

    mask = values.notna() & weights.notna()

    if mask.sum() == 0:
        return np.nan

    if weights.loc[mask].sum() == 0:
        return np.nan

    indicator = (values.loc[mask] == category_value).astype(float)

    return np.average(
        indicator,
        weights=weights.loc[mask]
    )


def count_valid_cases(data, value_col, weight_col):
    return (
        data[value_col].notna()
        & data[weight_col].notna()
    ).sum()


def sum_valid_weights(data, value_col, weight_col):
    mask = (
        data[value_col].notna()
        & data[weight_col].notna()
    )

    if mask.sum() == 0:
        return np.nan

    return data.loc[mask, weight_col].sum()


def kish_effective_n(data, value_col, weight_col):
    """
    Kish effective sample size for the same valid respondents used
    in the corresponding weighted estimate.

    n_eff = (sum w)^2 / sum(w^2)
    """
    mask = data[value_col].notna() & data[weight_col].notna()

    if mask.sum() == 0:
        return np.nan

    weights = pd.to_numeric(
        data.loc[mask, weight_col],
        errors="coerce"
    ).dropna()

    if len(weights) == 0:
        return np.nan

    denominator = np.square(weights.to_numpy(dtype=float)).sum()

    if denominator <= 0:
        return np.nan

    return float(weights.sum() ** 2 / denominator)



In [16]:
def get_activity_suffixes_from_files(file_info_list):
    suffixes = set()

    for file_info in file_info_list:
        file_path = Path(file_info["path"])

        columns = pd.read_csv(
            file_path,
            nrows=0
        ).columns

        for col in columns:
            if col.startswith(MONTHS12_PREFIX):
                suffix = col[len(MONTHS12_PREFIX):]
                suffixes.add(suffix)

            if col.startswith(DAYS10P60GR_PREFIX):
                suffix = col[len(DAYS10P60GR_PREFIX):]
                suffixes.add(suffix)

    suffixes = sorted(suffixes)

    print(f"Total activity suffixes found: {len(suffixes)}")

    return suffixes


activity_suffixes = get_activity_suffixes_from_files(
    Q3_FILES
)

Total activity suffixes found: 125


activity level all

In [17]:
def make_one_year_overall_activity_level_panel(
    file_path,
    year_value,
    weight_col=WEIGHT_COL
):
    file_path = Path(file_path)

    df = pd.read_csv(file_path)
    df = standardise_base_columns(df)
    df = df.replace(MISSING_CODES, np.nan)

    df["year"] = year_value
    df["age_group"] = df["Age9"].map(AGE9_LABELS)

    df = df[
        (df["Age16plus"] == 1)
        & df["age_group"].notna()
    ].copy()

    rows = []

    group_cols = [
        "year",
        "LA_2023",
        "age_group",
    ]

    overall_col = "MEMS7GR_ALL"

    if overall_col not in df.columns:
        print(f"{overall_col} missing in year {year_value}")
        return pd.DataFrame()

    for group_values, group_data in df.groupby(group_cols, dropna=False):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        row = dict(zip(group_cols, group_values))

        row["overall_inactive_rate"] = weighted_category_rate(
            data=group_data,
            value_col=overall_col,
            category_value=0,
            weight_col=weight_col
        )

        row["overall_fairly_active_rate"] = weighted_category_rate(
            data=group_data,
            value_col=overall_col,
            category_value=1,
            weight_col=weight_col
        )

        row["overall_active_rate"] = weighted_category_rate(
            data=group_data,
            value_col=overall_col,
            category_value=2,
            weight_col=weight_col
        )

        row["n_overall_activity_level"] = count_valid_cases(
            data=group_data,
            value_col=overall_col,
            weight_col=weight_col
        )

        row["weighted_n_overall_activity_level"] = sum_valid_weights(
            data=group_data,
            value_col=overall_col,
            weight_col=weight_col
        )

        row["n_eff_overall_activity_level"] = kish_effective_n(
            data=group_data,
            value_col=overall_col,
            weight_col=weight_col
        )

        row["small_cell_overall_activity_level"] = (
            row["n_overall_activity_level"] < 30
        )

        rows.append(row)

    result = pd.DataFrame(rows)

    print(
        f"Year {year_value}: {len(result)} overall activity level rows"
    )

    return result

In [18]:
def make_overall_activity_level_panel(
    file_info_list,
    output_path,
    weight_col=WEIGHT_COL
):
    panels = []

    for file_info in file_info_list:
        one_year_panel = make_one_year_overall_activity_level_panel(
            file_path=file_info["path"],
            year_value=file_info["year"],
            weight_col=weight_col
        )

        panels.append(one_year_panel)

    panel_df = pd.concat(
        panels,
        ignore_index=True
    )

    panel_df = panel_df.sort_values(
        [
            "year",
            "LA_2023",
            "age_group",
        ]
    ).reset_index(drop=True)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    panel_df.to_csv(output_path, index=False)

    print(f"Done: {len(panel_df)} rows")
    print("Years:")
    print(panel_df["year"].value_counts().sort_index())

    return panel_df


q3_overall_activity_level_panel = make_overall_activity_level_panel(
    file_info_list=Q3_FILES,
    output_path=OUTPUT_DIR / "q3_age_overall_activity_level_panel.csv",
    weight_col=WEIGHT_COL
)

Year 1: 256 overall activity level rows
Year 2: 256 overall activity level rows
Year 3: 256 overall activity level rows
Year 4: 256 overall activity level rows
Year 5: 256 overall activity level rows
Year 6: 256 overall activity level rows
Year 7: 256 overall activity level rows
Year 8: 256 overall activity level rows
Done: 2048 rows
Years:
year
1    256
2    256
3    256
4    256
5    256
6    256
7    256
8    256
Name: count, dtype: int64


specific activity participation and level (full)

In [19]:
def make_one_year_activity_participation_level_panel(
    file_path,
    year_value,
    activity_suffixes,
    weight_col=WEIGHT_COL
):
    file_path = Path(file_path)

    df = pd.read_csv(file_path)
    df = standardise_base_columns(df)
    df = df.replace(MISSING_CODES, np.nan)

    df["year"] = year_value
    df["age_group"] = df["Age9"].map(AGE9_LABELS)

    df = df[
        (df["Age16plus"] == 1)
        & df["age_group"].notna()
    ].copy()

    rows = []

    group_cols = [
        "year",
        "LA_2023",
        "age_group",
    ]

    for group_values, group_data in df.groupby(group_cols, dropna=False):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        base_row = dict(zip(group_cols, group_values))

        for suffix in activity_suffixes:
            months12_col = MONTHS12_PREFIX + suffix
            days10p60gr_col = DAYS10P60GR_PREFIX + suffix
            mems7gr_col = MEMS7GR_PREFIX + suffix

            row = base_row.copy()
            row["activity_suffix"] = suffix

            row["months12_available"] = months12_col in df.columns
            row["days10p60gr_available"] = days10p60gr_col in df.columns
            row["activity_level_available"] = mems7gr_col in df.columns

            if row["months12_available"]:
                row["months12_rate"] = weighted_rate_equal(
                    data=group_data,
                    value_col=months12_col,
                    positive_value=1,
                    weight_col=weight_col
                )

                row["n_months12"] = count_valid_cases(
                    data=group_data,
                    value_col=months12_col,
                    weight_col=weight_col
                )

                row["weighted_n_months12"] = sum_valid_weights(
                    data=group_data,
                    value_col=months12_col,
                    weight_col=weight_col
                )

                row["n_eff_months12"] = kish_effective_n(
                    data=group_data,
                    value_col=months12_col,
                    weight_col=weight_col
                )
            else:
                row["months12_rate"] = np.nan
                row["n_months12"] = 0
                row["weighted_n_months12"] = np.nan
                row["n_eff_months12"] = np.nan

            if row["days10p60gr_available"]:
                row["days10p60gr_rate"] = weighted_rate_positive(
                    data=group_data,
                    value_col=days10p60gr_col,
                    weight_col=weight_col
                )

                row["n_days10p60gr"] = count_valid_cases(
                    data=group_data,
                    value_col=days10p60gr_col,
                    weight_col=weight_col
                )

                row["weighted_n_days10p60gr"] = sum_valid_weights(
                    data=group_data,
                    value_col=days10p60gr_col,
                    weight_col=weight_col
                )

                row["n_eff_days10p60gr"] = kish_effective_n(
                    data=group_data,
                    value_col=days10p60gr_col,
                    weight_col=weight_col
                )
            else:
                row["days10p60gr_rate"] = np.nan
                row["n_days10p60gr"] = 0
                row["weighted_n_days10p60gr"] = np.nan
                row["n_eff_days10p60gr"] = np.nan

            if row["activity_level_available"]:
                row["activity_inactive_rate"] = weighted_category_rate(
                    data=group_data,
                    value_col=mems7gr_col,
                    category_value=0,
                    weight_col=weight_col
                )

                row["activity_fairly_active_rate"] = weighted_category_rate(
                    data=group_data,
                    value_col=mems7gr_col,
                    category_value=1,
                    weight_col=weight_col
                )

                row["activity_active_rate"] = weighted_category_rate(
                    data=group_data,
                    value_col=mems7gr_col,
                    category_value=2,
                    weight_col=weight_col
                )

                row["n_activity_level"] = count_valid_cases(
                    data=group_data,
                    value_col=mems7gr_col,
                    weight_col=weight_col
                )

                row["weighted_n_activity_level"] = sum_valid_weights(
                    data=group_data,
                    value_col=mems7gr_col,
                    weight_col=weight_col
                )

                row["n_eff_activity_level"] = kish_effective_n(
                    data=group_data,
                    value_col=mems7gr_col,
                    weight_col=weight_col
                )
            else:
                row["activity_inactive_rate"] = np.nan
                row["activity_fairly_active_rate"] = np.nan
                row["activity_active_rate"] = np.nan
                row["n_activity_level"] = 0
                row["weighted_n_activity_level"] = np.nan
                row["n_eff_activity_level"] = np.nan

            row["small_cell_months12"] = row["n_months12"] < 30
            row["small_cell_days10p60gr"] = row["n_days10p60gr"] < 30
            row["small_cell_activity_level"] = row["n_activity_level"] < 30

            rows.append(row)

    result = pd.DataFrame(rows)

    print(
        f"Year {year_value}: {len(result)} activity-specific rows"
    )

    return result

In [20]:
def make_activity_participation_level_panel(
    file_info_list,
    activity_suffixes,
    output_path,
    weight_col=WEIGHT_COL
):
    panels = []

    for file_info in file_info_list:
        one_year_panel = make_one_year_activity_participation_level_panel(
            file_path=file_info["path"],
            year_value=file_info["year"],
            activity_suffixes=activity_suffixes,
            weight_col=weight_col
        )

        panels.append(one_year_panel)

    panel_df = pd.concat(
        panels,
        ignore_index=True
    )

    panel_df = panel_df.sort_values(
        [
            "year",
            "LA_2023",
            "age_group",
            "activity_suffix",
        ]
    ).reset_index(drop=True)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    panel_df.to_csv(output_path, index=False)

    print(f"Done: {len(panel_df)} rows")
    print("Years:")
    print(panel_df["year"].value_counts().sort_index())

    print("Activities:")
    print(panel_df["activity_suffix"].nunique())

    return panel_df


q3_activity_participation_level_panel_full = make_activity_participation_level_panel(
    file_info_list=Q3_FILES,
    activity_suffixes=activity_suffixes,
    output_path=OUTPUT_DIR / "q3_age_activity_participation_level_panel_full.csv",
    weight_col=WEIGHT_COL
)

Year 1: 32000 activity-specific rows
Year 2: 32000 activity-specific rows
Year 3: 32000 activity-specific rows
Year 4: 32000 activity-specific rows
Year 5: 32000 activity-specific rows
Year 6: 32000 activity-specific rows
Year 7: 32000 activity-specific rows
Year 8: 32000 activity-specific rows
Done: 256000 rows
Years:
year
1    32000
2    32000
3    32000
4    32000
5    32000
6    32000
7    32000
8    32000
Name: count, dtype: int64
Activities:
125


check if activity suffix available

In [21]:
def make_activity_availability_summary(panel_df, output_path):
    rows = []

    for suffix in sorted(panel_df["activity_suffix"].dropna().unique()):
        activity_data = panel_df[
            panel_df["activity_suffix"] == suffix
        ].copy()

        year_data = activity_data[
            [
                "year",
                "months12_available",
                "days10p60gr_available",
                "activity_level_available",
            ]
        ].drop_duplicates()

        row = {}
        row["activity_suffix"] = suffix
        row["years_in_panel"] = year_data["year"].nunique()

        row["months12_years_available"] = year_data[
            year_data["months12_available"] == True
        ]["year"].nunique()

        row["days10p60gr_years_available"] = year_data[
            year_data["days10p60gr_available"] == True
        ]["year"].nunique()

        row["activity_level_years_available"] = year_data[
            year_data["activity_level_available"] == True
        ]["year"].nunique()

        row["complete_for_participation"] = (
            row["years_in_panel"] == 8
            and row["months12_years_available"] == 8
            and row["days10p60gr_years_available"] == 8
        )

        row["complete_for_participation_and_level"] = (
            row["years_in_panel"] == 8
            and row["months12_years_available"] == 8
            and row["days10p60gr_years_available"] == 8
            and row["activity_level_years_available"] == 8
        )

        rows.append(row)

    result = pd.DataFrame(rows)

    result.to_csv(output_path, index=False)

    print(f"Done: {len(result)} activities")

    print("Complete for participation:")
    print(result["complete_for_participation"].value_counts(dropna=False))

    print("Complete for participation and level:")
    print(result["complete_for_participation_and_level"].value_counts(dropna=False))

    return result


q3_activity_availability_summary = make_activity_availability_summary(
    panel_df=q3_activity_participation_level_panel_full,
    output_path=OUTPUT_DIR / "q3_age_activity_availability_summary.csv"
)

Done: 125 activities
Complete for participation:
complete_for_participation
True     124
False      1
Name: count, dtype: int64
Complete for participation and level:
complete_for_participation_and_level
True     124
False      1
Name: count, dtype: int64


specific activity participation and level (complete)

In [22]:
def make_complete_activity_participation_level_panel(
    panel_df,
    availability_df,
    output_path
):
    complete_activities = availability_df[
        availability_df["complete_for_participation_and_level"] == True
    ]["activity_suffix"].tolist()

    complete_df = panel_df[
        panel_df["activity_suffix"].isin(complete_activities)
    ].copy()

    complete_df = complete_df.sort_values(
        [
            "year",
            "LA_2023",
            "age_group",
            "activity_suffix",
        ]
    ).reset_index(drop=True)

    complete_df.to_csv(output_path, index=False)

    print(f"Complete activities: {len(complete_activities)}")
    print(f"Rows in complete panel: {len(complete_df)}")

    return complete_df


q3_activity_participation_level_panel_complete = make_complete_activity_participation_level_panel(
    panel_df=q3_activity_participation_level_panel_full,
    availability_df=q3_activity_availability_summary,
    output_path=OUTPUT_DIR / "q3_age_activity_participation_level_panel_complete.csv"
)

Complete activities: 124
Rows in complete panel: 253952


final check

In [23]:
print("Overall activity level panel:")
print(q3_overall_activity_level_panel.shape)

print("Activity-specific full panel:")
print(q3_activity_participation_level_panel_full.shape)

print("Activity-specific complete panel:")
print(q3_activity_participation_level_panel_complete.shape)

print("Overall years:")
print(q3_overall_activity_level_panel["year"].value_counts().sort_index())

print("Full panel years:")
print(q3_activity_participation_level_panel_full["year"].value_counts().sort_index())

print("Complete panel years:")
print(q3_activity_participation_level_panel_complete["year"].value_counts().sort_index())

Overall activity level panel:
(2048, 10)
Activity-specific full panel:
(256000, 24)
Activity-specific complete panel:
(253952, 24)
Overall years:
year
1    256
2    256
3    256
4    256
5    256
6    256
7    256
8    256
Name: count, dtype: int64
Full panel years:
year
1    32000
2    32000
3    32000
4    32000
5    32000
6    32000
7    32000
8    32000
Name: count, dtype: int64
Complete panel years:
year
1    31744
2    31744
3    31744
4    31744
5    31744
6    31744
7    31744
8    31744
Name: count, dtype: int64


In [24]:
print("Complete panel availability:")
print(q3_activity_participation_level_panel_complete[
    [
        "months12_available",
        "days10p60gr_available",
        "activity_level_available",
    ]
].drop_duplicates())

Complete panel availability:
   months12_available  days10p60gr_available  activity_level_available
0                True                   True                      True


check small cell

In [25]:
def print_small_cell_summary(df, small_cell_cols):
    for col in small_cell_cols:
        print("\nColumn:", col)
        print(df[col].value_counts(dropna=False))
        print((df[col].value_counts(normalize=True, dropna=False) * 100).round(2))


print("Overall activity level small cells:")
print_small_cell_summary(
    q3_overall_activity_level_panel,
    [
        "small_cell_overall_activity_level",
    ]
)


print("Activity-specific complete panel small cells:")
print_small_cell_summary(
    q3_activity_participation_level_panel_complete,
    [
        "small_cell_months12",
        "small_cell_days10p60gr",
        "small_cell_activity_level",
    ]
)

Overall activity level small cells:

Column: small_cell_overall_activity_level
small_cell_overall_activity_level
False    1604
True      444
Name: count, dtype: int64
small_cell_overall_activity_level
False    78.32
True     21.68
Name: proportion, dtype: float64
Activity-specific complete panel small cells:

Column: small_cell_months12
small_cell_months12
False    187972
True      65980
Name: count, dtype: int64
small_cell_months12
False    74.02
True     25.98
Name: proportion, dtype: float64

Column: small_cell_days10p60gr
small_cell_days10p60gr
False    187972
True      65980
Name: count, dtype: int64
small_cell_days10p60gr
False    74.02
True     25.98
Name: proportion, dtype: float64

Column: small_cell_activity_level
small_cell_activity_level
False    195027
True      58925
Name: count, dtype: int64
small_cell_activity_level
False    76.8
True     23.2
Name: proportion, dtype: float64


In [26]:

required_overall = {
    "n_overall_activity_level",
    "weighted_n_overall_activity_level",
    "n_eff_overall_activity_level",
}
required_activity = {
    "n_months12",
    "weighted_n_months12",
    "n_eff_months12",
    "n_days10p60gr",
    "weighted_n_days10p60gr",
    "n_eff_days10p60gr",
    "n_activity_level",
    "weighted_n_activity_level",
    "n_eff_activity_level",
}

missing_overall = sorted(required_overall - set(q3_overall_activity_level_panel.columns))
missing_activity = sorted(required_activity - set(q3_activity_participation_level_panel_complete.columns))

if missing_overall or missing_activity:
    raise RuntimeError(
        f"Missing required columns. Overall={missing_overall}; Activity={missing_activity}"
    )

checks = [
    (q3_overall_activity_level_panel,
     "n_eff_overall_activity_level", "n_overall_activity_level"),
    (q3_activity_participation_level_panel_complete,
     "n_eff_months12", "n_months12"),
    (q3_activity_participation_level_panel_complete,
     "n_eff_days10p60gr", "n_days10p60gr"),
    (q3_activity_participation_level_panel_complete,
     "n_eff_activity_level", "n_activity_level"),
]

for frame, neff_col, n_col in checks:
    bad = frame[
        frame[neff_col].notna()
        & frame[n_col].notna()
        & (frame[neff_col] > frame[n_col] + 1e-8)
    ]
    if not bad.empty:
        raise RuntimeError(
            f"{neff_col} exceeds {n_col} in {len(bad)} rows."
        )

print("Kish n_eff verification PASSED.")
print("\nOverall:")
print(q3_overall_activity_level_panel[
    ["n_overall_activity_level",
     "weighted_n_overall_activity_level",
     "n_eff_overall_activity_level"]
].describe())

print("\nActivity outcomes:")
print(q3_activity_participation_level_panel_complete[
    ["n_eff_months12",
     "n_eff_days10p60gr",
     "n_eff_activity_level"]
].describe())


Kish n_eff verification PASSED.

Overall:
       n_overall_activity_level  weighted_n_overall_activity_level  \
count               2048.000000                        2048.000000   
mean                  65.512695                         109.922694   
std                   41.102881                          74.614288   
min                    1.000000                           0.685869   
25%                   34.000000                          53.969611   
50%                   67.000000                         100.785840   
75%                   90.000000                         154.042951   
max                  252.000000                         481.394237   

       n_eff_overall_activity_level  
count                   2048.000000  
mean                      41.753973  
std                       24.725621  
min                        1.000000  
25%                       23.754559  
50%                       41.327896  
75%                       56.741062  
max                    